## 8 Ball Table Analyses - Task 1 Computer Vision

In [ ]:
import os
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math
import colorsys


In [ ]:
IMG_DIR = Path("development_set/")
OUTPUT_DIR= Path("output/top_views")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_EXTENSIONS = {".jpg",".jpeg", ".png", ".bmp"}
image_paths = sorted([ path for path in IMG_DIR.iterdir()
                       if path.suffix.lower() in VALID_EXTENSIONS])

print(f"Found {len(image_paths)} images")
for i, path in enumerate(image_paths[:5]):
    print(f"[{i}] {path.name}")


In [ ]:
def show_images_grid(images, titles=None, figsize_per_row=(18, 5)):
    if isinstance(images, np.ndarray):
        images = [images]

    if isinstance(titles, str):
        titles = [titles]

    num_images = len(images)

    if num_images == 0:
        print("No images provided to display.")
        return

    cols = min(3, num_images)

    rows = math.ceil(num_images / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(figsize_per_row[0], figsize_per_row[1] * rows))

    if hasattr(axes, 'flatten'):
        axes = axes.flatten()
    else:
        axes = [axes]

    for i, ax in enumerate(axes):
        if i < num_images:
            img = images[i]

            if len(img.shape) == 3:
                img_to_show = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                ax.imshow(img_to_show)
            else:
                ax.imshow(img, cmap="gray")

            if titles and i < len(titles):
                ax.set_title(titles[i])
            else:
                ax.set_title(f"Image {i}")

            ax.axis("off")

        else:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
if not image_paths:
    raise FileNotFoundError(f"No image files found in {IMG_DIR}")

paths_to_load = image_paths[:9]

loaded_images = []
image_titles = []

for path in paths_to_load:
    img = cv2.imread(str(path))

    if img is None:
        print(f"Could not load image: {path}")
        continue

    loaded_images.append(img)
    image_titles.append(f"Original: {path.name}")

    print(f"Loaded: {path.name} | Shape: {img.shape}")

if not loaded_images:
    raise FileNotFoundError("Failed to load any of the selected images.")

show_images_grid(loaded_images, titles=image_titles)

# Approach 1 - Only Using Color

In [ ]:
def hsl_to_bgr(h, s, l):
    h = h / 360.0
    s = s / 100.0
    l = l / 100.0

    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return (int(b * 255), int(g * 255), int(r * 255))

In [ ]:
def isolate_table_color(image, target_hsl=(206, 66, 59), tolerance=20):
    target_bgr = hsl_to_bgr(*target_hsl)

    lower = np.array([
        max(target_bgr[0] - tolerance, 0),
        max(target_bgr[1] - tolerance, 0),
        max(target_bgr[2] - tolerance, 0)
    ], dtype=np.uint8)

    upper = np.array([
        min(target_bgr[0] + tolerance, 255),
        min(target_bgr[1] + tolerance, 255),
        min(target_bgr[2] + tolerance, 255)
    ], dtype=np.uint8)

    mask = cv2.inRange(image, lower, upper)

    return mask

In [ ]:
COLOR_TABLE = (206, 66, 59)  # H, S, L

tolerance = 20

my_masks = []

for img in loaded_images:
    single_mask = isolate_table_color(img,COLOR_TABLE,tolerance)

    my_masks.append(single_mask)

show_images_grid(my_masks, image_titles)

In [ ]:
cols = 4
num_pairs = len(loaded_images)
total_images = num_pairs * 2

rows = math.ceil(total_images / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, 4* rows))

if hasattr(axes, 'flatten'):
    axes = axes.flatten()
else:
    axes = [axes]

for i in range(num_pairs):
    orig_idx = i * 2
    mask_idx = i * 2 + 1

    img = loaded_images[i]
    if len(img.shape) == 3:
        img_to_show = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[orig_idx].imshow(img_to_show)
    else:
        axes[orig_idx].imshow(img, cmap="gray")

    axes[orig_idx].set_title(f"Original {i}")
    axes[orig_idx].axis("off")

    mask = my_masks[i]
    axes[mask_idx].imshow(mask, cmap="gray")
    axes[mask_idx].set_title(f"Mask {i}")
    axes[mask_idx].axis("off")

for j in range(total_images, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def get_table_contour(binary_mask):
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        print("No contours found in the mask!")
        return None

    table_contour = max(contours, key=cv2.contourArea)
    return table_contour

In [ ]:
contour_images = []
contour_titles = []
my_contours = []

for i, (img, mask) in enumerate(zip(loaded_images, my_masks)):
    contour_img = img.copy()
    table_contour = get_table_contour(mask)

    my_contours.append(table_contour)

    if table_contour is not None:
        cv2.drawContours(contour_img, [table_contour], -1, (0, 255, 0), 3)
        contour_titles.append(f"Table Contour {i+1}")
    else:
        contour_titles.append(f"No Contour {i+1}")

    contour_images.append(contour_img)

show_images_grid(contour_images, titles=contour_titles)

In [ ]:
def order_points(pts):
    pts = np.array(pts, dtype="float32")
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)

    top_left = pts[np.argmin(s)]
    bottom_right = pts[np.argmax(s)]
    top_right = pts[np.argmin(diff)]
    bottom_left = pts[np.argmax(diff)]

    return np.array([top_left, top_right, bottom_right, bottom_left], dtype="float32")

In [ ]:
def get_table_corners(table_contour, padding=60):
    rot_rect = cv2.minAreaRect(table_contour)
    box = cv2.boxPoints(rot_rect)

    ordered_corners = order_points(box)

    ordered_corners[2][0] += padding
    ordered_corners[3][0] -= padding

    return ordered_corners

In [ ]:
corner_images = []
corner_titles = []
my_corners = []

for i, (img, contour) in enumerate(zip(loaded_images, my_contours)):
    corner_img = img.copy()

    if contour is not None:
        ordered_box = get_table_corners(contour, padding=60)

        my_corners.append(ordered_box)


        draw_box = np.intp(ordered_box)
        cv2.drawContours(corner_img, [draw_box], -1, (0, 0, 255), 3)

        corner_titles.append(f"Padded Corners {i+1}")
    else:
        my_corners.append(None)
        corner_titles.append(f"No Table {i+1}")

    corner_images.append(corner_img)

show_images_grid(corner_images, titles=corner_titles)

In [ ]:
def get_top_view(image, src_corners, width=800, height=600):
    dst_corners = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(src_corners, dst_corners)
    top_view = cv2.warpPerspective(image, M, (width, height))

    return top_view

In [ ]:
top_view_images = []
top_view_titles = []

for i, (img, corners) in enumerate(zip(loaded_images, my_corners)):

    if corners is not None:
        flat_table = get_top_view(img, corners, width=800, height=600)

        top_view_images.append(flat_table)
        top_view_titles.append(f"Top View {i+1}")
    else:
        top_view_images.append(np.zeros((600, 800, 3), dtype=np.uint8))
        top_view_titles.append(f"Failed {i+1}")

show_images_grid(top_view_images, titles=top_view_titles)

# Approach 2 - Using Lines